In [1]:
import os, json, uuid, httpx
from pathlib import Path
from datetime import datetime
 
import pytesseract
import cv2
import numpy as np
from PIL import Image

In [2]:
try:
    from pdf2image import convert_from_path
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️  pdf2image not installed — PDF support disabled. Install poppler for Windows.")

In [3]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [4]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"
OUTPUT_DIR   = Path("ocr_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [5]:
def load_document(file_path: str) -> list:
    path = Path(file_path)
    ext  = path.suffix.lower()
 
    if ext == ".pdf":
        if not PDF_SUPPORT:
            raise RuntimeError("pdf2image not available. Install poppler.")
        # poppler_path needed on Windows — adjust if needed
        # Download from: https://github.com/oschwartz10612/poppler-windows/releases
        # Then set: poppler_path=r"C:\poppler\Library\bin"
        pages = convert_from_path(str(path), dpi=300, poppler_path=r"C:\poppler\poppler-25.12.0\Library\bin")
        print(f"  📄 PDF loaded: {len(pages)} page(s)")
        return pages
 
    elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
        img = Image.open(path).convert("RGB")
        print(f"  🖼️  Image loaded: {img.size}")
        return [img]
 
    else:
        raise ValueError(f"Unsupported file type: {ext}")
 

In [6]:
def preprocess(pil_img: Image.Image) -> Image.Image:
    """
    Grayscale → Denoise → Adaptive threshold
    Makes text sharp and clean for Tesseract
    """
    # PIL → OpenCV
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray   = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
 
    # Denoise
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
 
    # Adaptive threshold — handles uneven lighting in scans
    thresh = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
 
    return Image.fromarray(thresh)
 

In [7]:
def run_ocr(pages: list) -> str:
    """Runs Tesseract on all pages, returns combined raw text."""
    config = "--oem 3 --psm 6"
    texts  = []
 
    for i, page in enumerate(pages):
        processed = preprocess(page)
        text      = pytesseract.image_to_string(processed, config=config)
        texts.append(text)
        print(f"  📝 Page {i+1}: {len(text)} chars extracted")
 
    return "\n".join(texts)
 

In [8]:
PROMPTS = {
 
    "PAN_CARD": """
Extract fields from this Company PAN Card OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "pan_number":    "10-char PAN e.g. AAKCM1234C",
  "entity_name":   "Full company name",
  "date_of_reg":   "DD/MM/YYYY",
  "entity_type":   "COMPANY or INDIVIDUAL",
  "issuing_auth":  "Income Tax Department"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "GST_CERTIFICATE": """
Extract fields from this GST Registration Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "gstin":              "15-char GSTIN",
  "pan_number":         "10-char PAN",
  "legal_name":         "Legal name of business",
  "trade_name":         "Trade name if different",
  "state_code":         "2-digit state code",
  "state":              "State name",
  "status":             "ACTIVE or CANCELLED",
  "registration_date":  "DD/MM/YYYY",
  "constitution":       "Private Limited etc",
  "address":            "Full address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "LEI_CERTIFICATE": """
Extract fields from this LEI Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "lei_code":           "20-char LEI",
  "legal_name":         "Legal name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "status":             "ACTIVE or INACTIVE",
  "registration_date":  "DD/MM/YYYY",
  "renewal_date":       "DD/MM/YYYY",
  "issuing_lou":        "Issuing organization",
  "country":            "Country"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "INCORPORATION_CERTIFICATE": """
Extract fields from this Certificate of Incorporation OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "date_of_incorp":     "DD/MM/YYYY",
  "company_type":       "Private Limited etc",
  "authorized_capital": "Amount in numbers only e.g. 2500000",
  "state":              "State name",
  "roc":                "ROC office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "MOA": """
Extract fields from this Memorandum of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "state":              "State of registered office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN",
  "authorized_capital": "Amount in numbers only",
  "main_objects":       ["list of main business objects"],
  "subscribers": [
    {"name": "subscriber name", "shares": "number of shares"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "AOA": """
Extract fields from this Articles of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "authorized_capital": "Amount in numbers only",
  "min_directors":      "minimum number",
  "max_directors":      "maximum number",
  "directors": [
    {"name": "director name", "din": "DIN number"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "REGISTERED_ADDRESS": """
Extract fields from this Registered Address / INC-22 OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":    "Full company name",
  "cin":             "CIN number",
  "pan_number":      "PAN number",
  "gstin":           "GSTIN",
  "lei":             "LEI code",
  "address_line1":   "Building/flat",
  "address_line2":   "Street/road",
  "area":            "Area/locality",
  "city":            "City",
  "state":           "State",
  "pincode":         "6-digit PIN",
  "srn":             "Service request number",
  "filing_date":     "DD/MM/YYYY",
  "approval_date":   "DD/MM/YYYY",
  "status":          "APPROVED or PENDING"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "ELECTRICITY_BILL": """
Extract fields from this Electricity Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "consumer_name":    "Name on bill",
  "consumer_number":  "Consumer/account number",
  "discom":           "Electricity provider name",
  "address":          "Consumer address",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "units_consumed":   "Number only",
  "total_amount":     "Number only e.g. 4693.70",
  "connection_type":  "Commercial or Residential"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "TELEPHONE_BILL": """
Extract fields from this Telephone/Landline Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "account_name":     "Name on bill",
  "account_number":   "Account number",
  "telephone_number": "Phone number",
  "provider":         "Telecom provider name",
  "address":          "Address on bill",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "total_amount":     "Number only",
  "connection_type":  "Landline or Mobile"
}
 
OCR TEXT:
{ocr_text}
"""
}

In [9]:
def extract_with_llm(ocr_text: str, doc_type: str) -> dict:
    """Sends OCR text to Qwen3 via Ollama, returns parsed JSON."""
 
    prompt_template = PROMPTS.get(doc_type)
    if not prompt_template:
        raise ValueError(f"No prompt defined for doc_type: {doc_type}")
 
    prompt = prompt_template.replace("{ocr_text}", ocr_text)
 
    print(f"  🤖 Sending to Qwen3...")
 
    try:
        response = httpx.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                # Tell Qwen3 to think less, just extract — faster
                "options": {
                    "temperature": 0.1,
                    "top_p": 0.9,
                }
            },
            timeout=120.0  # Qwen3 can be slow on CPU
        )
        response.raise_for_status()
        raw_response = response.json()["response"]
 
        # ── Clean up response ─────────────────────────────────
        # Qwen3 sometimes wraps in ```json ... ```
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        cleaned = cleaned.strip()
 
        # Also strip <think>...</think> blocks if present
        if "<think>" in cleaned:
            import re
            cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
 
        parsed = json.loads(cleaned)
        print(f"  ✅ JSON extracted: {len(parsed)} fields")
        return parsed
 
    except httpx.ConnectError:
        print("  ❌ Ollama not running. Start with: ollama serve")
        return {"error": "OLLAMA_NOT_RUNNING"}
    except json.JSONDecodeError as e:
        print(f"  ⚠️  JSON parse failed: {e}")
        print(f"  Raw response: {raw_response[:200]}")
        return {"error": "JSON_PARSE_FAILED", "raw": raw_response[:500]}
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {"error": str(e)}
 

In [10]:
def build_document_json(
    file_path: str,
    doc_type:  str,
    fields:    dict,
    ocr_text:  str
) -> dict:
    """Wraps extracted fields in the standard compliance JSON envelope."""
 
    # Cross-check fields — same keys across ALL documents
    # Used for consistency verification later
    cross_check = {
        "pan_number":   fields.get("pan_number")   or fields.get("pan"),
        "company_name": fields.get("entity_name")  or fields.get("legal_name")   or
                        fields.get("company_name") or fields.get("consumer_name") or
                        fields.get("account_name"),
        "address":      fields.get("address")      or
                        f"{fields.get('address_line1','')} {fields.get('address_line2','')}".strip(),
        "pincode":      fields.get("pincode"),
        "cin":          fields.get("cin"),
        "gstin":        fields.get("gstin"),
    }
 
    # Remove None values
    cross_check = {k: v for k, v in cross_check.items() if v}
 
    return {
        "doc_id":       str(uuid.uuid4())[:8].upper(),
        "doc_type":     doc_type,
        "source_file":  Path(file_path).name,
        "extracted_at": datetime.now().isoformat(),
        "fields":       fields,
        "cross_check":  cross_check,
        "ocr_raw_text": ocr_text[:1000],  # first 1000 chars for debug
        "status":       "FAILED" if "error" in fields else "EXTRACTED"
    }

In [11]:
def process_document(file_path: str, doc_type: str) -> dict:
    """Full pipeline for a single document."""
 
    print(f"\n{'─'*55}")
    print(f"  Processing: {Path(file_path).name}")
    print(f"  Doc Type  : {doc_type}")
    print(f"{'─'*55}")
 
    # Step 1 — Load
    pages = load_document(file_path)
 
    # Step 2+3 — Preprocess + OCR
    ocr_text = run_ocr(pages)
 
    # Step 4 — LLM extraction
    fields = extract_with_llm(ocr_text, doc_type)
 
    # Step 5 — Build final JSON
    doc_json = build_document_json(file_path, doc_type, fields, ocr_text)
 
    # Save to output folder
    out_file = OUTPUT_DIR / f"{doc_type}_{doc_json['doc_id']}.json"
    with open(out_file, "w") as f:
        json.dump(doc_json, f, indent=2)
 
    print(f"  💾 Saved: {out_file}")
    return doc_json

In [12]:
DOCUMENTS = [
    ("01_Company_PAN_Card.png",              "PAN_CARD"),
    ("02_GST_Certificate.pdf",               "GST_CERTIFICATE"),
    ("03_LEI_Certificate.pdf",               "LEI_CERTIFICATE"),
    ("04_Certificate_of_Incorporation.pdf",  "INCORPORATION_CERTIFICATE"),
    ("05_MOA.pdf",                           "MOA"),
    ("06_AOA.pdf",                           "AOA"),
    ("07_Registered_Address_INC22.pdf",      "REGISTERED_ADDRESS"),
    ("08_Electricity_Bill.pdf",              "ELECTRICITY_BILL"),
    ("09_Telephone_Landline_Bill.pdf",       "TELEPHONE_BILL"),
]
 


In [13]:
def normalize(value: str) -> str:
    """Normalize string for comparison — lowercase, strip extra spaces."""
    if not value:
        return ""
    return " ".join(str(value).lower().strip().split())

In [14]:
def run_cross_checks(results: list) -> dict:
    """
    Full cross-check engine.
    Checks 6 critical parameters across all documents that should contain them.
    Pinpoints exactly which document has the inconsistent value.
    """
 
    # Parameters to check → their key in cross_check block
    CHECKS = {
        "PAN Number":   "pan_number",
        "Company Name": "company_name",
        "Pincode":      "pincode",
        "CIN":          "cin",
        "GSTIN":        "gstin",
        "Address":      "address",
    }
 
    # Which doc types are expected to have each field
    FIELD_PRESENCE = {
        "pan_number": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "company_name": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "MOA", "AOA",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "pincode": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "cin": [
            "LEI_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "MOA", "AOA", "REGISTERED_ADDRESS"
        ],
        "gstin": [
            "GST_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "address": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
    }
 
    report = {"passed": [], "failed": [], "warnings": []}
 
    print("\n" + "=" * 65)
    print("   CROSS-CHECK REPORT")
    print("=" * 65)
 
    for label, key in CHECKS.items():
        expected_docs = FIELD_PRESENCE.get(key, [])
 
        found        = {}  # doc_type -> raw value
        missing_from = []
 
        for r in results:
            if r["doc_type"] not in expected_docs:
                continue
            val = r["cross_check"].get(key)
            if val:
                found[r["doc_type"]] = val
            else:
                missing_from.append(r["doc_type"])
 
        print(f"\n  {'-' * 60}")
        print(f"  Checking : {label}")
 
        if not found:
            msg = "Not extracted from any document — possible OCR failure"
            print(f"  WARNING  : {msg}")
            report["warnings"].append({"parameter": label, "issue": msg})
            continue
 
        normalized  = {doc: normalize(val) for doc, val in found.items()}
        unique_vals = set(normalized.values())
 
        if len(unique_vals) == 1:
            print(f"  RESULT   : CONSISTENT  ->  {list(found.values())[0]}")
            for doc in found:
                print(f"     {doc:<44} OK")
            report["passed"].append({
                "parameter": label,
                "value":     list(found.values())[0],
                "docs":      list(found.keys())
            })
 
        else:
            print(f"  RESULT   : MISMATCH DETECTED")
 
            # Group docs by their normalized value
            value_groups = {}
            for doc, val in found.items():
                norm = normalize(val)
                value_groups.setdefault(norm, []).append((doc, val))
 
            # Majority = most docs agree on this value
            majority_norm = max(value_groups, key=lambda x: len(value_groups[x]))
            majority_val  = value_groups[majority_norm][0][1]
 
            for norm_val, doc_list in value_groups.items():
                is_majority = (norm_val == majority_norm)
                tag = "majority (expected)" if is_majority else "INCONSISTENT"
                for doc, raw_val in doc_list:
                    print(f"     {doc:<44} {tag}")
                    print(f"       Value: \"{raw_val}\"")
 
            inconsistent_docs = [
                doc
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for doc, _ in doc_list
            ]
            inconsistent_vals = [
                val
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for _, val in doc_list
            ]
 
            print(f"\n  ACTION REQUIRED:")
            print(f"    Kindly check the following document(s) : {inconsistent_docs}")
            print(f"    Check consistency for                  : [{label}]")
            print(f"    Inconsistent value(s) found            : {inconsistent_vals}")
            print(f"    Expected value (majority)              : \"{majority_val}\"")
 
            report["failed"].append({
                "parameter":           label,
                "majority_value":      majority_val,
                "inconsistent_docs":   inconsistent_docs,
                "inconsistent_values": inconsistent_vals,
                "all_values":          found
            })
 
        if missing_from:
            print(f"  WARNING  : Field not extracted from (possible OCR issue): {missing_from}")
            report["warnings"].append({
                "parameter":    label,
                "missing_from": missing_from,
                "issue":        "Expected but not extracted — verify manually"
            })
 
    # Final verdict
    print(f"\n  {'=' * 65}")
    print(f"  FINAL VERDICT")
    print(f"  {'=' * 65}")
    print(f"  Passed   : {len(report['passed'])} parameter(s)")
    print(f"  Failed   : {len(report['failed'])} parameter(s)")
    print(f"  Warnings : {len(report['warnings'])} parameter(s)")
 
    if report["failed"]:
        print(f"\n  INCONSISTENCIES FOUND — Review before proceeding:")
        for item in report["failed"]:
            print(f"    [{item['parameter']}]  ->  inconsistent in: {item['inconsistent_docs']}")
        print(f"\n  Status: COMPLIANCE FAILED — Manual review required")
    elif report["warnings"]:
        print(f"\n  Status: PARTIAL — Some fields missing, verify manually")
    else:
        print(f"\n  Status: ALL CHECKS PASSED — Documents are consistent")
 
    print(f"  {'=' * 65}")
    return report
 


In [15]:
def run_all():
    print("\n" + "=" * 65)
    print("   NBFC OCR PIPELINE — Processing all documents")
    print("=" * 65)
 
    results = []
    failed  = []
 
    for filename, doc_type in DOCUMENTS:
        if not Path(filename).exists():
            print(f"\n  File not found: {filename} — skipping")
            failed.append(filename)
            continue
        result = process_document(filename, doc_type)
        results.append(result)
 
    # Processing summary
    print("\n" + "=" * 65)
    print("   PROCESSING SUMMARY")
    print("=" * 65)
    print(f"  Total Documents : {len(DOCUMENTS)}")
    print(f"  Processed       : {len(results)}")
    print(f"  Not Found       : {len(failed)}")
 
    extracted = [r for r in results if r["status"] == "EXTRACTED"]
    errors    = [r for r in results if r["status"] == "FAILED"]
    print(f"  Extracted OK    : {len(extracted)}")
    print(f"  LLM Errors      : {len(errors)}")
 
    if errors:
        print(f"\n  Documents with extraction errors:")
        for e in errors:
            print(f"    {e['doc_type']} ({e['source_file']})")
 
    # Run full cross-check
    cross_check_report = run_cross_checks(results)
 
    # Save both outputs
    master_file = OUTPUT_DIR / "all_documents.json"
    with open(master_file, "w") as f:
        json.dump(results, f, indent=2)
 
    report_file = OUTPUT_DIR / "cross_check_report.json"
    with open(report_file, "w") as f:
        json.dump(cross_check_report, f, indent=2)
 
    print(f"\n  Master JSON    : {master_file}")
    print(f"  Cross-Check    : {report_file}")
    print("=" * 65)
 
    return results, cross_check_report
 

In [16]:
if __name__ == "__main__":
    run_all()
 


   NBFC OCR PIPELINE — Processing all documents

───────────────────────────────────────────────────────
  Processing: 01_Company_PAN_Card.png
  Doc Type  : PAN_CARD
───────────────────────────────────────────────────────
  🖼️  Image loaded: (1012, 638)
  📝 Page 1: 278 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 5 fields
  💾 Saved: ocr_output\PAN_CARD_FED2B17A.json

───────────────────────────────────────────────────────
  Processing: 02_GST_Certificate.pdf
  Doc Type  : GST_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1184 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 11 fields
  💾 Saved: ocr_output\GST_CERTIFICATE_63717978.json

───────────────────────────────────────────────────────
  Processing: 03_LEI_Certificate.pdf
  Doc Type  : LEI_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1073 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSO

---
# 🔐 API VERIFICATION LAYER
### Stage 2 — Verify extracted fields against real + sandbox APIs

**APIs used:**
| Field | Source | Type |
|---|---|---|
| LEI Code | GLEIF api.gleif.org | ✅ Real, free forever |
| Pincode | api.postalpincode.in | ✅ Real, free forever |
| PAN format | Local regex | ✅ No API needed |
| GSTIN format | Local regex | ✅ No API needed |
| CIN format | Local regex | ✅ No API needed |
| Company name match | difflib fuzzy | ✅ No API needed |
| PAN active/inactive | Sandbox mock | ✅ Simulates NSDL |
| GST active/inactive | Sandbox mock | ✅ Simulates GST Portal |
| CIN/MCA status | Sandbox mock | ✅ Simulates MCA21 |

In [17]:
# ── Load OCR results from previous pipeline run ───────────
import json, re, httpx, difflib
from pathlib import Path
from datetime import datetime, timedelta

OCR_OUTPUT = Path("ocr_output")

def load_ocr_results() -> list:
    master = OCR_OUTPUT / "all_documents.json"
    if not master.exists():
        raise FileNotFoundError("Run the OCR pipeline first — all_documents.json not found")
    with open(master) as f:
        docs = json.load(f)
    print(f"✅ Loaded {len(docs)} document(s) from OCR output")
    for d in docs:
        print(f"   {d['doc_type']:<40} status={d['status']}")
    return docs

OCR_DOCS = load_ocr_results()

# Helper — get extracted field from a specific doc type
def get_field(doc_type: str, field: str):
    for d in OCR_DOCS:
        if d['doc_type'] == doc_type:
            return d.get('fields', {}).get(field) or d.get('cross_check', {}).get(field)
    return None

# Pull key fields extracted by OCR
EXTRACTED = {
    'pan':     get_field('PAN_CARD', 'pan_number')     or get_field('GST_CERTIFICATE', 'pan_number'),
    'gstin':   get_field('GST_CERTIFICATE', 'gstin'),
    'lei':     get_field('LEI_CERTIFICATE', 'lei_code'),
    'cin':     get_field('INCORPORATION_CERTIFICATE', 'cin'),
    'pincode': get_field('REGISTERED_ADDRESS', 'pincode') or get_field('GST_CERTIFICATE', 'pincode'),
    'company': get_field('GST_CERTIFICATE', 'legal_name'),
    'bill_date': get_field('ELECTRICITY_BILL', 'bill_date'),
}

print("\n📋 Key fields for verification:")
for k, v in EXTRACTED.items():
    print(f"   {k:<12} : {v}")

✅ Loaded 9 document(s) from OCR output
   PAN_CARD                                 status=EXTRACTED
   GST_CERTIFICATE                          status=EXTRACTED
   LEI_CERTIFICATE                          status=EXTRACTED
   INCORPORATION_CERTIFICATE                status=EXTRACTED
   MOA                                      status=EXTRACTED
   AOA                                      status=EXTRACTED
   REGISTERED_ADDRESS                       status=EXTRACTED
   ELECTRICITY_BILL                         status=EXTRACTED
   TELEPHONE_BILL                           status=EXTRACTED

📋 Key fields for verification:
   pan          : AKCM1234C
   gstin        : 15-char GSTIN
   lei          : 20-char LEI
   cin          : U45200TN2015PTC123456
   pincode      : 600018
   company      : MANIKANDAN CONSTRUCTIONS PVT LTD
   bill_date    : 20/11/2024


In [18]:
# ═══════════════════════════════════════════════════════════
# SANDBOX — Simulates NSDL, GST Portal, MCA21
# In production: swap these calls with real API endpoints
# ═══════════════════════════════════════════════════════════

SANDBOX = {
    'pan': {
        'AAKCM1234C': {'name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'ACTIVE',   'type': 'Company'},
        'BNZPM2501F': {'name': 'D MANIKANDAN',                     'status': 'ACTIVE',   'type': 'Individual'},
        'ZZZZZ9999Z': {'name': 'SHELL CORP LTD',                   'status': 'INACTIVE', 'type': 'Company'},
    },
    'gst': {
        '33AAKCM1234C1ZP': {'legal_name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'Active',    'state': 'Tamil Nadu', 'state_code': '33', 'reg_date': '15/04/2015'},
        '27ZZZZZ9999Z1Z5': {'legal_name': 'SHELL CORP LTD',                   'status': 'Cancelled', 'state': 'Maharashtra'},
    },
    'cin': {
        'U45200TN2015PTC123456': {'company_name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'Active', 'roc': 'ROC Chennai', 'incorp_date': '15/04/2015'},
        'U45200MH2010PTC999999': {'company_name': 'SHELL CORP LTD',                   'status': 'Strike Off', 'roc': 'ROC Mumbai'},
    }
}

def sandbox_pan(pan: str) -> dict:
    data = SANDBOX['pan'].get(pan)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'pan': pan}
    return {'verified': data['status'] == 'ACTIVE', 'status': data['status'],
            'name': data['name'], 'type': data['type'], 'source': 'SANDBOX', 'pan': pan}

def sandbox_gst(gstin: str) -> dict:
    data = SANDBOX['gst'].get(gstin)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'gstin': gstin}
    return {'verified': data['status'] == 'Active', 'status': data['status'],
            'legal_name': data['legal_name'], 'state': data['state'],
            'state_code': data['state_code'], 'source': 'SANDBOX', 'gstin': gstin}

def sandbox_cin(cin: str) -> dict:
    data = SANDBOX['cin'].get(cin)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'cin': cin}
    return {'verified': data['status'] == 'Active', 'status': data['status'],
            'company_name': data['company_name'], 'roc': data['roc'],
            'incorp_date': data['incorp_date'], 'source': 'SANDBOX', 'cin': cin}

print('✅ Sandbox DB ready (PAN, GST, CIN)')

✅ Sandbox DB ready (PAN, GST, CIN)


In [19]:
# ═══════════════════════════════════════════════════════════
# FORMAT VALIDATORS — local, no API needed
# ═══════════════════════════════════════════════════════════

def validate_pan_format(pan: str) -> dict:
    """PAN: 5 letters, 4 digits, 1 letter. 4th char = entity type."""
    if not pan:
        return {'valid': False, 'error': 'PAN is None/empty'}
    pan = pan.strip().upper()
    pattern = re.compile(r'^[A-Z]{5}[0-9]{4}[A-Z]$')
    if not pattern.match(pan):
        return {'valid': False, 'pan': pan, 'error': 'Invalid PAN format'}
    entity_map = {'P': 'Individual', 'C': 'Company', 'H': 'HUF', 'F': 'Firm',
                  'A': 'AOP', 'T': 'Trust', 'B': 'BOI', 'G': 'Govt', 'J': 'AJP'}
    entity = entity_map.get(pan[3], 'Unknown')
    return {'valid': True, 'pan': pan, 'entity_type_code': pan[3],
            'entity_type': entity, 'is_company': pan[3] == 'C'}

def validate_gstin_format(gstin: str) -> dict:
    """GSTIN: 15 chars. State code (2) + PAN (10) + entity (1) + Z + checksum."""
    if not gstin:
        return {'valid': False, 'error': 'GSTIN is None/empty'}
    gstin = gstin.strip().upper()
    pattern = re.compile(r'^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$')
    if not pattern.match(gstin):
        return {'valid': False, 'gstin': gstin, 'error': 'Invalid GSTIN format'}
    state_codes = {
        '01':'Jammu & Kashmir','02':'Himachal Pradesh','03':'Punjab','04':'Chandigarh',
        '05':'Uttarakhand','06':'Haryana','07':'Delhi','08':'Rajasthan','09':'Uttar Pradesh',
        '10':'Bihar','11':'Sikkim','12':'Arunachal Pradesh','13':'Nagaland','14':'Manipur',
        '15':'Mizoram','16':'Tripura','17':'Meghalaya','18':'Assam','19':'West Bengal',
        '20':'Jharkhand','21':'Odisha','22':'Chhattisgarh','23':'Madhya Pradesh',
        '24':'Gujarat','26':'Daman & Diu','27':'Maharashtra','29':'Karnataka',
        '30':'Goa','31':'Lakshadweep','32':'Kerala','33':'Tamil Nadu','34':'Puducherry',
        '35':'Andaman & Nicobar','36':'Telangana','37':'Andhra Pradesh'
    }
    sc   = gstin[:2]
    pan  = gstin[2:12]
    state = state_codes.get(sc, 'Unknown')
    return {'valid': True, 'gstin': gstin, 'state_code': sc,
            'state': state, 'embedded_pan': pan, 'format_ok': True}

def validate_cin_format(cin: str) -> dict:
    """CIN: U/L + 5 digits + 2 state letters + 4 year + 3 type letters + 6 digits."""
    if not cin:
        return {'valid': False, 'error': 'CIN is None/empty'}
    cin = cin.strip().upper()
    pattern = re.compile(r'^[UL][0-9]{5}[A-Z]{2}[0-9]{4}[A-Z]{3}[0-9]{6}$')
    if not pattern.match(cin):
        return {'valid': False, 'cin': cin, 'error': 'Invalid CIN format'}
    return {'valid': True, 'cin': cin,
            'listing_status': 'Listed' if cin[0] == 'L' else 'Unlisted',
            'industry_code': cin[1:6],
            'state_code': cin[6:8],
            'year_of_incorp': cin[8:12],
            'company_type': cin[12:15]}

def validate_lei_format(lei: str) -> dict:
    """LEI: 20 alphanumeric characters."""
    if not lei:
        return {'valid': False, 'error': 'LEI is None/empty'}
    lei = lei.strip().upper()
    if not re.match(r'^[A-Z0-9]{20}$', lei):
        return {'valid': False, 'lei': lei, 'error': 'Invalid LEI format (must be 20 alphanumeric chars)'}
    return {'valid': True, 'lei': lei, 'lou_prefix': lei[:4]}

# Test all format validators
print('=== FORMAT VALIDATION TEST ===')
print('PAN :', validate_pan_format(EXTRACTED.get('pan', '')))
print('GST :', validate_gstin_format(EXTRACTED.get('gstin', '')))
print('CIN :', validate_cin_format(EXTRACTED.get('cin', '')))
print('LEI :', validate_lei_format(EXTRACTED.get('lei', '')))

=== FORMAT VALIDATION TEST ===
PAN : {'valid': False, 'pan': 'AKCM1234C', 'error': 'Invalid PAN format'}
GST : {'valid': False, 'gstin': '15-CHAR GSTIN', 'error': 'Invalid GSTIN format'}
CIN : {'valid': True, 'cin': 'U45200TN2015PTC123456', 'listing_status': 'Unlisted', 'industry_code': '45200', 'state_code': 'TN', 'year_of_incorp': '2015', 'company_type': 'PTC'}
LEI : {'valid': False, 'lei': '20-CHAR LEI', 'error': 'Invalid LEI format (must be 20 alphanumeric chars)'}


In [20]:
# ═══════════════════════════════════════════════════════════
# PAN ↔ GSTIN CROSS-EMBED CHECK
# GSTIN chars 3–12 must equal PAN. No API needed.
# ═══════════════════════════════════════════════════════════

def check_pan_gstin_embed(pan: str, gstin: str) -> dict:
    """PAN embedded in GSTIN at positions 3–12 must match."""
    if not pan or not gstin:
        return {'consistent': False, 'error': 'PAN or GSTIN missing'}
    pan   = pan.strip().upper()
    gstin = gstin.strip().upper()
    embedded = gstin[2:12]
    match = embedded == pan
    return {
        'consistent':    match,
        'pan':           pan,
        'gstin':         gstin,
        'embedded_pan':  embedded,
        'message':       'PAN correctly embedded in GSTIN' if match else f'MISMATCH: PAN={pan} but GSTIN embeds={embedded}'
    }

result = check_pan_gstin_embed(EXTRACTED.get('pan'), EXTRACTED.get('gstin'))
print('PAN ↔ GSTIN Embed Check:')
print(f"  Consistent : {result['consistent']}")
print(f"  Message    : {result['message']}")

PAN ↔ GSTIN Embed Check:
  Consistent : False
  Message    : MISMATCH: PAN=AKCM1234C but GSTIN embeds=-CHAR GSTI


In [21]:
# ═══════════════════════════════════════════════════════════
# GLEIF — Real Free API. No key, no registration, forever free.
# https://api.gleif.org/api/v1/lei-records/{LEI}
# ═══════════════════════════════════════════════════════════

def verify_lei_gleif(lei: str, expected_name: str = None) -> dict:
    """Verify LEI against GLEIF. Completely free, no auth."""
    if not lei:
        return {'verified': False, 'error': 'No LEI provided', 'source': 'GLEIF'}
    lei = lei.strip().upper()
    url = f'https://api.gleif.org/api/v1/lei-records/{lei}'
    try:
        resp = httpx.get(url, timeout=10.0)
        if resp.status_code == 404:
            return {'verified': False, 'status': 'NOT_FOUND', 'lei': lei, 'source': 'GLEIF'}
        if resp.status_code != 200:
            return {'verified': False, 'error': f'HTTP {resp.status_code}', 'source': 'GLEIF'}
        data        = resp.json()['data']['attributes']
        reg         = data['registration']
        entity      = data['entity']
        legal_name  = entity['legalName']['name']
        status      = reg['status']
        renewal     = reg.get('nextRenewalDate', 'N/A')
        # Name match check
        name_match = True
        if expected_name:
            similarity = difflib.SequenceMatcher(None, legal_name.lower(), expected_name.lower()).ratio()
            name_match = similarity > 0.80
        return {
            'verified':         status == 'ISSUED',
            'lei':              lei,
            'legal_name':       legal_name,
            'status':           status,
            'next_renewal':     renewal,
            'corroboration':    reg.get('corroborationLevel', 'N/A'),
            'name_match':       name_match,
            'source':           'GLEIF'
        }
    except httpx.ConnectError:
        # Fallback if no internet — use format check only
        fmt = validate_lei_format(lei)
        return {'verified': fmt['valid'], 'lei': lei,
                'note': 'No internet — format check only', 'source': 'GLEIF_OFFLINE'}
    except Exception as e:
        return {'verified': False, 'error': str(e), 'source': 'GLEIF'}

lei_result = verify_lei_gleif(EXTRACTED.get('lei'), EXTRACTED.get('company'))
print('GLEIF LEI Verification:')
for k, v in lei_result.items():
    print(f'  {k:<20} : {v}')

GLEIF LEI Verification:
  verified             : False
  status               : NOT_FOUND
  lei                  : 20-CHAR LEI
  source               : GLEIF


In [22]:
# ═══════════════════════════════════════════════════════════
# INDIA PINCODE API — Free, no auth
# https://api.postalpincode.in/pincode/{pincode}
# ═══════════════════════════════════════════════════════════

def verify_pincode(pincode: str, expected_state: str = None, expected_city: str = None) -> dict:
    """Verify Indian pincode. Free, no API key."""
    if not pincode:
        return {'verified': False, 'error': 'No pincode provided'}
    pincode = str(pincode).strip()
    if not re.match(r'^[1-9][0-9]{5}$', pincode):
        return {'verified': False, 'pincode': pincode, 'error': 'Invalid pincode format'}
    try:
        resp = httpx.get(f'https://api.postalpincode.in/pincode/{pincode}', timeout=8.0)
        data = resp.json()
        if not data or data[0]['Status'] != 'Success':
            return {'verified': False, 'pincode': pincode, 'error': 'Pincode not found'}
        post_offices = data[0]['PostOffice']
        if not post_offices:
            return {'verified': False, 'pincode': pincode, 'error': 'No post offices found'}
        po    = post_offices[0]
        state = po.get('State', '')
        dist  = po.get('District', '')
        # State match
        state_match = True
        if expected_state:
            state_match = expected_state.lower() in state.lower() or state.lower() in expected_state.lower()
        return {
            'verified':     True,
            'pincode':      pincode,
            'state':        state,
            'district':     dist,
            'region':       po.get('Region', ''),
            'post_offices': len(post_offices),
            'state_match':  state_match,
            'source':       'INDIA_POSTAL_API'
        }
    except httpx.ConnectError:
        return {'verified': re.match(r'^[1-9][0-9]{5}$', pincode) is not None,
                'pincode': pincode, 'note': 'No internet — format check only'}
    except Exception as e:
        return {'verified': False, 'error': str(e)}

pin_result = verify_pincode(EXTRACTED.get('pincode'), expected_state='Tamil Nadu')
print('Pincode Verification:')
for k, v in pin_result.items():
    print(f'  {k:<20} : {v}')

Pincode Verification:
  verified             : True
  pincode              : 600018
  state                : Tamil Nadu
  district             : Chennai
  region               : Chennai Region
  post_offices         : 2
  state_match          : True
  source               : INDIA_POSTAL_API


In [23]:
# ═══════════════════════════════════════════════════════════
# SANDBOX API CALLS — PAN, GST, CIN
# Simulates NSDL, GST Portal, MCA21
# Production: swap function body with real httpx.get/post calls
# ═══════════════════════════════════════════════════════════

pan_result = sandbox_pan(EXTRACTED.get('pan', ''))
gst_result = sandbox_gst(EXTRACTED.get('gstin', ''))
cin_result = sandbox_cin(EXTRACTED.get('cin', ''))

print('PAN  (Sandbox/NSDL)  :', pan_result)
print('GST  (Sandbox/Portal):', gst_result)
print('CIN  (Sandbox/MCA21) :', cin_result)

PAN  (Sandbox/NSDL)  : {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'pan': 'AKCM1234C'}
GST  (Sandbox/Portal): {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'gstin': '15-char GSTIN'}
CIN  (Sandbox/MCA21) : {'verified': True, 'status': 'Active', 'company_name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'roc': 'ROC Chennai', 'incorp_date': '15/04/2015', 'source': 'SANDBOX', 'cin': 'U45200TN2015PTC123456'}


In [24]:
# ═══════════════════════════════════════════════════════════
# COMPANY NAME FUZZY MATCH
# Checks if names across all API responses are consistent.
# Handles 'PVT LTD' vs 'PRIVATE LIMITED' type variations.
# ═══════════════════════════════════════════════════════════

def normalize_company_name(name: str) -> str:
    if not name: return ''
    n = name.upper().strip()
    replacements = [
        ('PRIVATE LIMITED', 'PVT LTD'),
        ('PVT. LTD.', 'PVT LTD'),
        ('PVT.LTD.', 'PVT LTD'),
        ('LIMITED', 'LTD'),
        ('LTD.', 'LTD'),
    ]
    for old, new in replacements:
        n = n.replace(old, new)
    return ' '.join(n.split())

def fuzzy_name_match(names: dict, threshold: float = 0.82) -> dict:
    """
    names = {'source_label': 'Company Name'}
    Returns consistency check across all sources.
    """
    normalized = {src: normalize_company_name(n) for src, n in names.items() if n}
    results    = {}
    base_src, base_name = list(normalized.items())[0]
    all_match = True
    for src, name in normalized.items():
        score = difflib.SequenceMatcher(None, base_name, name).ratio()
        match = score >= threshold
        if not match: all_match = False
        results[src] = {'name': name, 'score': round(score, 3), 'match': match}
    return {'consistent': all_match, 'base': base_name, 'details': results, 'threshold': threshold}

# Collect names from all sources
names_to_check = {
    'OCR_PAN':        get_field('PAN_CARD', 'entity_name'),
    'OCR_GST':        get_field('GST_CERTIFICATE', 'legal_name'),
    'OCR_LEI':        get_field('LEI_CERTIFICATE', 'legal_name'),
    'OCR_INCORP':     get_field('INCORPORATION_CERTIFICATE', 'company_name'),
    'SANDBOX_PAN':    pan_result.get('name'),
    'SANDBOX_GST':    gst_result.get('legal_name'),
    'SANDBOX_CIN':    cin_result.get('company_name'),
    'GLEIF':          lei_result.get('legal_name'),
}
# Filter None values
names_to_check = {k: v for k, v in names_to_check.items() if v}

name_result = fuzzy_name_match(names_to_check)
print(f"Company Name Consistency (threshold={name_result['threshold']})")
print(f"  Overall Consistent : {name_result['consistent']}")
print(f"  Base Name          : {name_result['base']}")
print()
for src, d in name_result['details'].items():
    mark = '✅' if d['match'] else '❌'
    print(f"  {mark} {src:<20} score={d['score']:.2f}  name={d['name']}")

Company Name Consistency (threshold=0.82)
  Overall Consistent : True
  Base Name          : MANIKANDAN CONSTRUCTIONS PVT LTD

  ✅ OCR_PAN              score=1.00  name=MANIKANDAN CONSTRUCTIONS PVT LTD
  ✅ OCR_GST              score=1.00  name=MANIKANDAN CONSTRUCTIONS PVT LTD
  ✅ OCR_LEI              score=1.00  name=MANIKANDAN CONSTRUCTIONS PVT LTD
  ✅ OCR_INCORP           score=1.00  name=MANIKANDAN CONSTRUCTIONS PVT LTD
  ✅ SANDBOX_CIN          score=1.00  name=MANIKANDAN CONSTRUCTIONS PVT LTD


In [25]:
# ═══════════════════════════════════════════════════════════
# BILL DATE VALIDATION
# Electricity/Telephone bills must be within 90 days.
# NBFC compliance standard.
# ═══════════════════════════════════════════════════════════

def validate_bill_date(bill_date_str: str, max_days: int = 90) -> dict:
    """Check if bill date is within allowed window (default 90 days)."""
    if not bill_date_str:
        return {'valid': False, 'error': 'No bill date found'}
    formats = ['%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d', '%d/%m/%y']
    parsed  = None
    for fmt in formats:
        try:
            parsed = datetime.strptime(bill_date_str.strip(), fmt)
            break
        except ValueError:
            continue
    if not parsed:
        return {'valid': False, 'error': f'Cannot parse date: {bill_date_str}'}
    today    = datetime.now()
    age_days = (today - parsed).days
    valid    = 0 <= age_days <= max_days
    return {
        'valid':      valid,
        'bill_date':  bill_date_str,
        'age_days':   age_days,
        'max_days':   max_days,
        'message':    f'Bill is {age_days} days old — {"WITHIN" if valid else "EXCEEDS"} {max_days}-day limit'
    }

elec_date = get_field('ELECTRICITY_BILL', 'bill_date')
tel_date  = get_field('TELEPHONE_BILL',  'bill_date')
print('Electricity Bill Date:', validate_bill_date(elec_date))
print('Telephone  Bill Date :', validate_bill_date(tel_date))

Electricity Bill Date: {'valid': False, 'bill_date': '20/11/2024', 'age_days': 500, 'max_days': 90, 'message': 'Bill is 500 days old — EXCEEDS 90-day limit'}
Telephone  Bill Date : {'valid': False, 'bill_date': '25/11/2024', 'age_days': 495, 'max_days': 90, 'message': 'Bill is 495 days old — EXCEEDS 90-day limit'}


In [26]:
# ═══════════════════════════════════════════════════════════
# MASTER VERIFICATION RUNNER
# Runs all checks and produces a final verification report.
# ═══════════════════════════════════════════════════════════

def run_api_verification() -> dict:
    print('\n' + '='*60)
    print('   API VERIFICATION REPORT')
    print('='*60)
    report = {'timestamp': datetime.now().isoformat(), 'checks': {}, 'passed': 0, 'failed': 0, 'warnings': 0}

    def add(name, result, critical=True):
        passed   = result.get('verified') or result.get('valid') or result.get('consistent') or False
        report['checks'][name] = {**result, 'critical': critical}
        if passed:
            report['passed']  += 1
            print(f'  ✅  {name}')
        else:
            if critical:
                report['failed'] += 1
                print(f'  ❌  {name}')
            else:
                report['warnings'] += 1
                print(f'  ⚠️   {name} (non-critical)')
        # Print key detail
        for key in ['error','message','status','state','note']:
            if result.get(key):
                print(f'       → {result[key]}')
                break

    print('\n  ── FORMAT CHECKS (local) ──────────────────────────')
    add('PAN Format',         validate_pan_format(EXTRACTED.get('pan',   '')))
    add('GSTIN Format',       validate_gstin_format(EXTRACTED.get('gstin','')))
    add('CIN Format',         validate_cin_format(EXTRACTED.get('cin',  '')))
    add('LEI Format',         validate_lei_format(EXTRACTED.get('lei',  '')))
    add('PAN ↔ GSTIN Embed',  check_pan_gstin_embed(EXTRACTED.get('pan'), EXTRACTED.get('gstin')))

    print('\n  ── REAL APIs ──────────────────────────────────────')
    add('LEI (GLEIF)',         verify_lei_gleif(EXTRACTED.get('lei'), EXTRACTED.get('company')))
    add('Pincode (Postal API)',verify_pincode(EXTRACTED.get('pincode'), expected_state='Tamil Nadu'))

    print('\n  ── SANDBOX (simulates NSDL, GST Portal, MCA21) ───')
    add('PAN Active (Sandbox)',sandbox_pan(EXTRACTED.get('pan', '')))
    add('GST Active (Sandbox)',sandbox_gst(EXTRACTED.get('gstin', '')))
    add('CIN Active (Sandbox)',sandbox_cin(EXTRACTED.get('cin', '')))

    print('\n  ── NAME CONSISTENCY ───────────────────────────────')
    names = {k: v for k, v in {
        'OCR_GST': get_field('GST_CERTIFICATE', 'legal_name'),
        'SANDBOX_PAN': sandbox_pan(EXTRACTED.get('pan','')).get('name'),
        'SANDBOX_GST': sandbox_gst(EXTRACTED.get('gstin','')).get('legal_name'),
        'GLEIF': verify_lei_gleif(EXTRACTED.get('lei'), None).get('legal_name'),
    }.items() if v}
    add('Company Name Consistency', fuzzy_name_match(names), critical=False)

    print('\n  ── BILL DATE CHECKS ───────────────────────────────')
    add('Electricity Bill Date', validate_bill_date(get_field('ELECTRICITY_BILL','bill_date')), critical=False)
    add('Telephone Bill Date',   validate_bill_date(get_field('TELEPHONE_BILL','bill_date')),   critical=False)

    # Final verdict
    print('\n' + '='*60)
    print(f"  ✅ Passed   : {report['passed']}")
    print(f"  ❌ Failed   : {report['failed']}")
    print(f"  ⚠️  Warnings : {report['warnings']}")
    if report['failed'] == 0:
        print('\n  VERDICT: ✅ ALL CRITICAL CHECKS PASSED')
        report['verdict'] = 'PASS'
    else:
        print(f"\n  VERDICT: ❌ {report['failed']} CRITICAL CHECK(S) FAILED — Do not proceed")
        report['verdict'] = 'FAIL'
    print('='*60)

    # Save report
    out = Path('ocr_output') / 'api_verification_report.json'
    with open(out, 'w') as f:
        json.dump(report, f, indent=2)
    print(f'\n  💾 Report saved: {out}')
    return report

VERIFICATION_REPORT = run_api_verification()


   API VERIFICATION REPORT

  ── FORMAT CHECKS (local) ──────────────────────────
  ❌  PAN Format
       → Invalid PAN format
  ❌  GSTIN Format
       → Invalid GSTIN format
  ✅  CIN Format
  ❌  LEI Format
       → Invalid LEI format (must be 20 alphanumeric chars)
  ❌  PAN ↔ GSTIN Embed
       → MISMATCH: PAN=AKCM1234C but GSTIN embeds=-CHAR GSTI

  ── REAL APIs ──────────────────────────────────────
  ❌  LEI (GLEIF)
       → NOT_FOUND
  ✅  Pincode (Postal API)
       → Tamil Nadu

  ── SANDBOX (simulates NSDL, GST Portal, MCA21) ───
  ❌  PAN Active (Sandbox)
       → NOT_FOUND
  ❌  GST Active (Sandbox)
       → NOT_FOUND
  ✅  CIN Active (Sandbox)
       → Active

  ── NAME CONSISTENCY ───────────────────────────────
  ✅  Company Name Consistency

  ── BILL DATE CHECKS ───────────────────────────────
  ⚠️   Electricity Bill Date (non-critical)
       → Bill is 500 days old — EXCEEDS 90-day limit
  ⚠️   Telephone Bill Date (non-critical)
       → Bill is 495 days old — EXCEEDS 90-day 

---
# 🔍 FRAUD DETECTION PIPELINE
### Stage 3 — KYC Points 8–12: Directors, KMPs, Beneficial Owners, PEP, RPT

**What this stage does:**
1. Generates fake test documents (Board, KMP, Beneficial Owners, PEP, RPT)
2. OCR extracts structured data from each doc using `qwen2.5:3b`
3. Builds entity relationship graph (who owns what, who directs what)
4. Computes beneficial ownership chains (direct + indirect %)
5. PEP check from extracted declarations
6. RPT risk scoring using rule-based weights
7. NLP risk summary via Qwen2.5:3b
8. Final risk score (0–100) + recommendation

In [27]:
# ═══════════════════════════════════════════════════════════
# GENERATE FRAUD TEST DOCUMENTS
# Fake but realistic PDFs for KYC Points 8-12
# ═══════════════════════════════════════════════════════════

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.pdfgen import canvas as rl_canvas

FRAUD_DIR = Path('fraud_test_docs')
FRAUD_DIR.mkdir(exist_ok=True)
W, H = A4

def _hdr(c, title, bg='#1a3c6e'):
    c.setFillColor(colors.HexColor(bg))
    c.rect(0, H-52, W, 52, fill=1, stroke=0)
    c.setFillColor(colors.white)
    c.setFont('Helvetica-Bold', 12)
    c.drawCentredString(W/2, H-24, title)
    c.setFont('Helvetica', 8.5)
    c.drawCentredString(W/2, H-40, 'MANIKANDAN CONSTRUCTIONS PVT LTD  |  FAKE DOCUMENT — OCR TESTING ONLY')

def _row(c, y, label, value):
    c.setFont('Helvetica-Bold', 8.5)
    c.setFillColor(colors.black)
    c.drawString(50, y, label)
    c.setFont('Helvetica', 8.5)
    c.drawString(210, y, f': {value}')
    c.setStrokeColor(colors.HexColor('#eeeeee'))
    c.setLineWidth(0.3)
    c.line(50, y-4, W-50, y-4)
    return y - 15

def _section(c, y, title, bg='#1a3c6e'):
    c.setFillColor(colors.HexColor(bg))
    c.rect(40, y-5, W-80, 17, fill=1, stroke=0)
    c.setFillColor(colors.white)
    c.setFont('Helvetica-Bold', 8.5)
    c.drawString(48, y, title)
    return y - 22

# ── DOC 1: Board of Directors ────────────────────────────────
def make_board_of_directors():
    path = str(FRAUD_DIR / '10_Board_of_Directors.pdf')
    c = rl_canvas.Canvas(path, pagesize=A4)
    _hdr(c, 'LIST OF BOARD OF DIRECTORS')
    y = H - 68
    c.setFont('Helvetica-Bold', 10)
    c.setFillColor(colors.HexColor('#1a3c6e'))
    c.drawString(50, y, 'Company: MANIKANDAN CONSTRUCTIONS PVT LTD')
    y -= 14
    c.setFont('Helvetica', 9)
    c.drawString(50, y, 'CIN: U45200TN2015PTC123456  |  PAN: AAKCM1234C  |  As of: 01/04/2024')
    y -= 28
    directors = [
        {'name': 'D MANIKANDAN',  'din': '06712345', 'pan': 'BNZPM2501F', 'dob': '16/07/1986',
         'designation': 'Managing Director', 'shareholding': '50%',
         'address': 'No.14, Anna Salai, Teynampet, Chennai - 600018',
         'nationality': 'Indian',
         'other_directorships': 'RAJESH INFRA PVT LTD (Director since 2018)'},
        {'name': 'S RAJESHWARI',  'din': '07823456', 'pan': 'ZZZZR8765K', 'dob': '22/03/1989',
         'designation': 'Whole-Time Director', 'shareholding': '50%',
         'address': 'No.28, T.Nagar, Chennai - 600017',
         'nationality': 'Indian',
         'other_directorships': 'NIL'},
    ]
    for i, d in enumerate(directors):
        c.setFillColor(colors.HexColor('#f0f4ff'))
        c.rect(40, y-118, W-80, 122, fill=1, stroke=0)
        c.setFillColor(colors.HexColor('#1a3c6e'))
        c.setFont('Helvetica-Bold', 10)
        c.drawString(50, y-10, f'Director {i+1}: {d["name"]}')
        y2 = y - 26
        for label, key in [
            ('DIN','din'),('PAN','pan'),('Date of Birth','dob'),
            ('Designation','designation'),('Shareholding','shareholding'),
            ('Address','address'),('Nationality','nationality'),
            ('Other Directorships','other_directorships')
        ]:
            y2 = _row(c, y2, label, d[key])
        y = y2 - 18
    c.setFont('Helvetica-Oblique', 8)
    c.setFillColor(colors.HexColor('#888888'))
    c.drawString(50, 35, 'FAKE DOCUMENT — FOR FRAUD DETECTION TESTING ONLY')
    c.save()
    return path

# ── DOC 2: KMP List ──────────────────────────────────────────
def make_kmp_list():
    path = str(FRAUD_DIR / '11_KMP_List.pdf')
    c = rl_canvas.Canvas(path, pagesize=A4)
    _hdr(c, 'LIST OF KEY MANAGERIAL PERSONS (KMPs)', bg='#2e5900')
    y = H - 68
    kmps = [
        {'name': 'D MANIKANDAN',   'designation': 'Managing Director (MD)',
         'id_numbers': 'DIN: 06712345 | PAN: BNZPM2501F',
         'email': 'md@manikandanconstructions.in', 'phone': '9884012345'},
        {'name': 'S RAJESHWARI',   'designation': 'Whole-Time Director (WTD)',
         'id_numbers': 'DIN: 07823456 | PAN: ZZZZR8765K',
         'email': 'wtd@manikandanconstructions.in', 'phone': '9884056789'},
        {'name': 'R SUBRAMANIAM',  'designation': 'Chief Financial Officer (CFO)',
         'id_numbers': 'PAN: BBBCS4321A',
         'email': 'cfo@manikandanconstructions.in', 'phone': '9884099001'},
        {'name': 'M PRIYA',        'designation': 'Company Secretary (CS)',
         'id_numbers': 'ACS No: A12345 | PAN: CCCMP5678B',
         'email': 'cs@manikandanconstructions.in', 'phone': '9884077654'},
    ]
    for kmp in kmps:
        c.setFillColor(colors.HexColor('#f0faf0'))
        c.rect(40, y-72, W-80, 76, fill=1, stroke=0)
        c.setFillColor(colors.HexColor('#2e5900'))
        c.setFont('Helvetica-Bold', 10)
        c.drawString(50, y-10, f"{kmp['name']}  —  {kmp['designation']}")
        y2 = y - 26
        for label, key in [('ID Numbers','id_numbers'),('Email','email'),('Phone','phone')]:
            y2 = _row(c, y2, label, kmp[key])
        y = y2 - 18
    c.setFont('Helvetica-Oblique', 8)
    c.setFillColor(colors.HexColor('#888888'))
    c.drawString(50, 35, 'FAKE DOCUMENT — FOR FRAUD DETECTION TESTING ONLY')
    c.save()
    return path

# ── DOC 3: Beneficial Owners ─────────────────────────────────
def make_beneficial_owners():
    path = str(FRAUD_DIR / '12_Beneficial_Owners.pdf')
    c = rl_canvas.Canvas(path, pagesize=A4)
    _hdr(c, 'UBO DECLARATION — BENEFICIAL OWNERS', bg='#4a0072')
    y = H - 68
    c.setFont('Helvetica', 9)
    c.setFillColor(colors.black)
    c.drawString(50, y, 'Pursuant to Companies (Significant Beneficial Owners) Rules, 2018')
    y -= 22
    # Direct UBOs
    y = _section(c, y, 'DIRECT BENEFICIAL OWNERS', bg='#4a0072')
    ubos = [
        {'name': 'D MANIKANDAN', 'pan': 'BNZPM2501F',
         'direct_holding': '50%', 'indirect_holding': '20% via DURAISAMY HOLDINGS',
         'total_effective': '70%', 'nature': 'Direct + Indirect Beneficial Owner',
         'declaration_date': '01/04/2024'},
        {'name': 'S RAJESHWARI', 'pan': 'ZZZZR8765K',
         'direct_holding': '50%', 'indirect_holding': 'NIL',
         'total_effective': '50%', 'nature': 'Direct Beneficial Owner',
         'declaration_date': '01/04/2024'},
    ]
    for o in ubos:
        c.setFillColor(colors.HexColor('#f5eeff'))
        c.rect(40, y-86, W-80, 90, fill=1, stroke=0)
        c.setFillColor(colors.HexColor('#4a0072'))
        c.setFont('Helvetica-Bold', 10)
        c.drawString(50, y-10, f"UBO: {o['name']}  |  PAN: {o['pan']}")
        y2 = y - 26
        for label, key in [
            ('Direct Holding','direct_holding'),('Indirect Holding','indirect_holding'),
            ('Total Effective %','total_effective'),('Nature','nature'),
            ('Declaration Date','declaration_date')
        ]:
            y2 = _row(c, y2, label, o[key])
        y = y2 - 18
    # Related entity
    y = _section(c, y, 'RELATED ENTITIES WITH INDIRECT STAKE', bg='#4a0072')
    c.setFillColor(colors.HexColor('#fff3e0'))
    c.rect(40, y-62, W-80, 66, fill=1, stroke=0)
    c.setFillColor(colors.HexColor('#e65100'))
    c.setFont('Helvetica-Bold', 10)
    c.drawString(50, y-10, 'Entity: DURAISAMY HOLDINGS')
    y2 = y - 26
    for label, val in [
        ('Entity PAN', 'DDDH7890C'),
        ('Relationship', 'D MANIKANDAN is Beneficial Owner of DURAISAMY HOLDINGS'),
        ('Ownership in Subject Company', '20% indirect stake'),
        ('Nature of Entity', 'Holding Company'),
    ]:
        y2 = _row(c, y2, label, val)
    c.setFont('Helvetica-Oblique', 8)
    c.setFillColor(colors.HexColor('#888888'))
    c.drawString(50, 35, 'FAKE DOCUMENT — FOR FRAUD DETECTION TESTING ONLY')
    c.save()
    return path

# ── DOC 4: PEP Declaration ───────────────────────────────────
def make_pep_declaration():
    path = str(FRAUD_DIR / '13_PEP_Declaration.pdf')
    c = rl_canvas.Canvas(path, pagesize=A4)
    _hdr(c, 'POLITICALLY EXPOSED PERSONS (PEP) DECLARATION', bg='#b71c1c')
    y = H - 68
    c.setFont('Helvetica', 9)
    c.setFillColor(colors.black)
    c.drawString(50, y,      'Pursuant to RBI KYC Master Directions 2016 (Updated 2023)')
    c.drawString(50, y-14,   'and FATF Recommendations on Politically Exposed Persons.')
    y -= 36
    persons = [
        {'name': 'D MANIKANDAN',  'designation': 'Managing Director',
         'pep_status': 'NOT A PEP',
         'family_pep': 'No immediate family member holds or has held a prominent public function',
         'associate_pep': 'No close associate is a Politically Exposed Person',
         'declaration': 'I hereby declare that I am not a Politically Exposed Person'},
        {'name': 'S RAJESHWARI', 'designation': 'Whole-Time Director',
         'pep_status': 'NOT A PEP',
         'family_pep': 'No immediate family member holds or has held a prominent public function',
         'associate_pep': 'No close associate is a Politically Exposed Person',
         'declaration': 'I hereby declare that I am not a Politically Exposed Person'},
    ]
    for p in persons:
        c.setFillColor(colors.HexColor('#ffeaea'))
        c.rect(40, y-90, W-80, 94, fill=1, stroke=0)
        c.setFillColor(colors.HexColor('#b71c1c'))
        c.setFont('Helvetica-Bold', 10)
        c.drawString(50, y-10, f"{p['name']}  —  {p['designation']}")
        y2 = y - 26
        for label, key in [
            ('PEP Status','pep_status'),('Family PEP','family_pep'),
            ('Associate PEP','associate_pep'),('Declaration','declaration')
        ]:
            y2 = _row(c, y2, label, p[key])
        y = y2 - 18
    c.setFont('Helvetica-Oblique', 8)
    c.setFillColor(colors.HexColor('#888888'))
    c.drawString(50, 35, 'FAKE DOCUMENT — FOR FRAUD DETECTION TESTING ONLY')
    c.save()
    return path

# ── DOC 5: RPT Document ──────────────────────────────────────
def make_rpt_document():
    path = str(FRAUD_DIR / '14_RPT_Related_Party_Transactions.pdf')
    c = rl_canvas.Canvas(path, pagesize=A4)
    _hdr(c, 'RELATED PARTY TRANSACTIONS DISCLOSURE', bg='#01579b')
    y = H - 68
    c.setFont('Helvetica', 9)
    c.setFillColor(colors.black)
    c.drawString(50, y, 'Pursuant to Section 188 of Companies Act 2013 and AS-18 Related Party Disclosures')
    y -= 24
    rpts = [
        {'related_party': 'RAJESH INFRA PVT LTD',
         'relationship': 'D MANIKANDAN is Director in both companies',
         'transaction_type': 'Sub-contracting of civil works',
         'amount': 'Rs. 45,00,000',
         'terms': 'At arms length, prevailing market rate',
         'approval': 'Board approval obtained on 01/04/2024',
         'risk_flag': 'MONITOR — common director, potential fund diversion risk'},
        {'related_party': 'DURAISAMY HOLDINGS',
         'relationship': 'D MANIKANDAN is Beneficial Owner; holds 20% indirect stake in subject company',
         'transaction_type': 'Inter-company deposit',
         'amount': 'Rs. 20,00,000',
         'terms': 'At arms length, 8% per annum',
         'approval': 'Shareholder approval obtained at EGM dated 15/03/2024',
         'risk_flag': 'LOW — standard holding company deposit, arms length'},
    ]
    for rpt in rpts:
        c.setFillColor(colors.HexColor('#e1f5fe'))
        c.rect(40, y-108, W-80, 112, fill=1, stroke=0)
        c.setFillColor(colors.HexColor('#01579b'))
        c.setFont('Helvetica-Bold', 10)
        c.drawString(50, y-10, f"Related Party: {rpt['related_party']}")
        y2 = y - 26
        for label, key in [
            ('Relationship','relationship'),('Transaction Type','transaction_type'),
            ('Amount','amount'),('Terms','terms'),('Board Approval','approval'),
            ('Risk Flag','risk_flag')
        ]:
            y2 = _row(c, y2, label, rpt[key])
        y = y2 - 18
    c.setFont('Helvetica-Oblique', 8)
    c.setFillColor(colors.HexColor('#888888'))
    c.drawString(50, 35, 'FAKE DOCUMENT — FOR FRAUD DETECTION TESTING ONLY')
    c.save()
    return path

print('Generating fraud test documents...')
board_path = make_board_of_directors()
kmp_path   = make_kmp_list()
benef_path = make_beneficial_owners()
pep_path   = make_pep_declaration()
rpt_path   = make_rpt_document()
print(f'  {board_path}')
print(f'  {kmp_path}')
print(f'  {benef_path}')
print(f'  {pep_path}')
print(f'  {rpt_path}')
print('All 5 fraud test docs ready.')

Generating fraud test documents...
  fraud_test_docs\10_Board_of_Directors.pdf
  fraud_test_docs\11_KMP_List.pdf
  fraud_test_docs\12_Beneficial_Owners.pdf
  fraud_test_docs\13_PEP_Declaration.pdf
  fraud_test_docs\14_RPT_Related_Party_Transactions.pdf
All 5 fraud test docs ready.


In [28]:
# ═══════════════════════════════════════════════════════════
# OCR PROMPTS FOR FRAUD DOCUMENTS
# Same qwen2.5:3b pipeline as compliance stage
# ═══════════════════════════════════════════════════════════

FRAUD_PROMPTS = {

    'BOARD_OF_DIRECTORS': '''
Extract all directors from this Board of Directors document.
Return ONLY valid JSON, no explanation, no markdown.
{
  "company_name": "",
  "cin": "",
  "pan": "",
  "directors": [
    {
      "name": "",
      "din": "",
      "pan": "",
      "dob": "DD/MM/YYYY",
      "designation": "",
      "shareholding": "%",
      "address": "",
      "nationality": "",
      "other_directorships": ""
    }
  ]
}
OCR TEXT:
{ocr_text}
''',

    'KMP_LIST': '''
Extract all Key Managerial Persons from this document.
Return ONLY valid JSON, no explanation, no markdown.
{
  "company_name": "",
  "kmps": [
    {
      "name": "",
      "designation": "",
      "id_numbers": "",
      "email": "",
      "phone": ""
    }
  ]
}
OCR TEXT:
{ocr_text}
''',

    'BENEFICIAL_OWNERS': '''
Extract beneficial ownership information from this UBO declaration.
Return ONLY valid JSON, no explanation, no markdown.
{
  "company_name": "",
  "ubos": [
    {
      "name": "",
      "pan": "",
      "direct_holding": "%",
      "indirect_holding": "",
      "total_effective": "%",
      "nature": ""
    }
  ],
  "related_entities": [
    {
      "name": "",
      "pan": "",
      "relationship": "",
      "ownership_pct": "%"
    }
  ]
}
OCR TEXT:
{ocr_text}
''',

    'PEP_DECLARATION': '''
Extract PEP declaration details from this document.
Return ONLY valid JSON, no explanation, no markdown.
{
  "company_name": "",
  "declarations": [
    {
      "name": "",
      "designation": "",
      "pep_status": "PEP or NOT A PEP",
      "family_pep": "",
      "associate_pep": ""
    }
  ]
}
OCR TEXT:
{ocr_text}
''',

    'RPT_DOCUMENT': '''
Extract Related Party Transaction details from this document.
Return ONLY valid JSON, no explanation, no markdown.
{
  "company_name": "",
  "related_party_transactions": [
    {
      "related_party": "",
      "relationship": "",
      "transaction_type": "",
      "amount": "",
      "terms": "",
      "approval": "",
      "risk_flag": ""
    }
  ]
}
OCR TEXT:
{ocr_text}
'''
}

FRAUD_DOCUMENT_LIST = [
    (str(FRAUD_DIR / '10_Board_of_Directors.pdf'),             'BOARD_OF_DIRECTORS'),
    (str(FRAUD_DIR / '11_KMP_List.pdf'),                      'KMP_LIST'),
    (str(FRAUD_DIR / '12_Beneficial_Owners.pdf'),             'BENEFICIAL_OWNERS'),
    (str(FRAUD_DIR / '13_PEP_Declaration.pdf'),               'PEP_DECLARATION'),
    (str(FRAUD_DIR / '14_RPT_Related_Party_Transactions.pdf'),'RPT_DOCUMENT'),
]

print(f'Prompts ready for {len(FRAUD_PROMPTS)} fraud document types')
print(f'Documents to process: {len(FRAUD_DOCUMENT_LIST)}')

Prompts ready for 5 fraud document types
Documents to process: 5


In [29]:
# ═══════════════════════════════════════════════════════════
# RUN OCR ON ALL FRAUD DOCUMENTS
# Uses load_document, run_ocr, extract_with_llm from Stage 1
# ═══════════════════════════════════════════════════════════

FRAUD_OCR_RESULTS = []

print('\n' + '='*55)
print('   OCR — FRAUD DOCUMENTS')
print('='*55)

for filepath, doc_type in FRAUD_DOCUMENT_LIST:
    print(f'\n  Processing: {Path(filepath).name}')
    print(f'  Doc Type  : {doc_type}')
    print('  ' + '-'*40)
    try:
        pages    = load_document(filepath)
        raw_text = run_ocr(pages)
        print(f'  Raw text: {len(raw_text)} chars')

        prompt = FRAUD_PROMPTS[doc_type].replace('{ocr_text}', raw_text)
        print('  Sending to Qwen2.5:3b...')
        resp = httpx.post(
            OLLAMA_URL,
            json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.1}},
            timeout=120.0
        )
        raw_resp = resp.json()['response'].strip()
        import re as _re
        cleaned  = _re.sub(r'<think>.*?</think>', '', raw_resp, flags=_re.DOTALL).strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('```')[1]
            if cleaned.startswith('json'): cleaned = cleaned[4:]
        parsed = json.loads(cleaned.strip())

        result = {
            'doc_type':    doc_type,
            'source_file': Path(filepath).name,
            'status':      'EXTRACTED',
            'fields':      parsed
        }
        FRAUD_OCR_RESULTS.append(result)

        out_file = OUTPUT_DIR / f'{doc_type}_{str(uuid.uuid4())[:6].upper()}.json'
        with open(out_file, 'w') as f:
            json.dump(result, f, indent=2)
        print(f'  Extracted {len(parsed)} top-level fields')
        print(f'  Saved: {out_file}')

    except json.JSONDecodeError as e:
        print(f'  JSON parse failed: {e}')
        FRAUD_OCR_RESULTS.append({'doc_type': doc_type, 'status': 'FAILED', 'error': 'JSON_PARSE_FAILED'})
    except Exception as e:
        print(f'  Error: {e}')
        FRAUD_OCR_RESULTS.append({'doc_type': doc_type, 'status': 'FAILED', 'error': str(e)})

extracted_ok = [r for r in FRAUD_OCR_RESULTS if r['status'] == 'EXTRACTED']
print(f'\n  Extracted OK : {len(extracted_ok)}/{len(FRAUD_DOCUMENT_LIST)}')


   OCR — FRAUD DOCUMENTS

  Processing: 10_Board_of_Directors.pdf
  Doc Type  : BOARD_OF_DIRECTORS
  ----------------------------------------
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 737 chars extracted
  Raw text: 737 chars
  Sending to Qwen2.5:3b...
  Extracted 4 top-level fields
  Saved: ocr_output\BOARD_OF_DIRECTORS_8FB8EB.json

  Processing: 11_KMP_List.pdf
  Doc Type  : KMP_LIST
  ----------------------------------------
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 708 chars extracted
  Raw text: 708 chars
  Sending to Qwen2.5:3b...
  Extracted 2 top-level fields
  Saved: ocr_output\KMP_LIST_093024.json

  Processing: 12_Beneficial_Owners.pdf
  Doc Type  : BENEFICIAL_OWNERS
  ----------------------------------------
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 859 chars extracted
  Raw text: 859 chars
  Sending to Qwen2.5:3b...
  Extracted 3 top-level fields
  Saved: ocr_output\BENEFICIAL_OWNERS_428032.json

  Processing: 13_PEP_Declaration.pdf
  Doc Type  : PEP_DECLARATION
  -------------------

In [30]:
# ═══════════════════════════════════════════════════════════
# ENTITY RELATIONSHIP GRAPH
# Builds directed graph from extracted fraud docs.
# Node types:  person | company | bank
# Edge types:  ownership | directorship | beneficial | financial
# ═══════════════════════════════════════════════════════════

import re as _re

def build_entity_graph(fraud_results: list) -> dict:
    nodes = {}  # id -> node dict
    edges = []  # list of edge dicts

    def add_node(nid, label, ntype, data=None):
        if nid not in nodes:
            nodes[nid] = {'id': nid, 'label': label, 'type': ntype,
                          'risk': 'low', 'data': data or {}}

    def add_edge(from_id, to_id, label, etype):
        duplicate = any(
            e['from'] == from_id and e['to'] == to_id and e['type'] == etype
            for e in edges
        )
        if not duplicate:
            edges.append({'from': from_id, 'to': to_id, 'label': label, 'type': etype})

    # Root node — the subject company applying for the loan
    add_node('MAIN_CO', 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'company',
             {'pan': 'AAKCM1234C', 'cin': 'U45200TN2015PTC123456'})

    for doc in fraud_results:
        if doc.get('status') != 'EXTRACTED':
            continue
        fields = doc.get('fields', {})

        if doc['doc_type'] == 'BOARD_OF_DIRECTORS':
            for director in fields.get('directors', []):
                din  = director.get('din', '')
                nid  = f"DIR_{din}" if din else f"DIR_{director.get('name','').replace(' ','_')}"
                add_node(nid, director.get('name', ''), 'person', {
                    'din': director.get('din'),
                    'pan': director.get('pan'),
                    'dob': director.get('dob'),
                    'designation': director.get('designation'),
                })
                add_edge(nid, 'MAIN_CO',
                         f"Director ({director.get('shareholding', '?')})", 'ownership')

                # Cross-directorships → new company nodes
                others = director.get('other_directorships', '')
                if others and others.upper() not in ('NIL', 'NONE', 'NA', ''):
                    companies = _re.findall(
                        r'([A-Z][A-Z\s&\.]+(?:PVT\.?\s*LTD|LIMITED|LLP|LTD)\.?)',
                        others.upper()
                    )
                    for comp in companies:
                        comp = comp.strip()
                        cid  = f"CO_{comp.replace(' ', '_')[:18]}"
                        add_node(cid, comp, 'company', {'note': 'Cross-directorship'})
                        nodes[cid]['risk'] = 'medium'
                        add_edge(nid, cid, 'Director', 'directorship')

        elif doc['doc_type'] == 'BENEFICIAL_OWNERS':
            for entity in fields.get('related_entities', []):
                pan  = entity.get('pan', '')
                eid  = f"ENT_{pan}" if pan else f"ENT_{entity.get('name','').replace(' ','_')[:18]}"
                add_node(eid, entity.get('name', ''), 'company', {
                    'pan': pan, 'relationship': entity.get('relationship')
                })
                add_edge(eid, 'MAIN_CO',
                         f"Owns {entity.get('ownership_pct', '?')}", 'ownership')

                # Link beneficial owner person to this holding entity
                for ubo in fields.get('ubos', []):
                    indirect = ubo.get('indirect_holding', '')
                    if entity.get('name', '').upper() in indirect.upper():
                        ubo_pan = ubo.get('pan', '')
                        uid = f"DIR_{ubo_pan}" if ubo_pan else f"DIR_{ubo.get('name','').replace(' ','_')}"
                        if uid in nodes:
                            add_edge(uid, eid, 'Beneficial Owner', 'beneficial')
                            nodes[uid]['risk'] = 'medium'

        elif doc['doc_type'] == 'RPT_DOCUMENT':
            for txn in fields.get('related_party_transactions', []):
                party = txn.get('related_party', '')
                rid   = f"RPT_{party.replace(' ', '_')[:18]}"
                if rid not in nodes:
                    add_node(rid, party, 'company', {'rpt': True})
                    nodes[rid]['risk'] = 'medium'
                add_edge('MAIN_CO', rid,
                         txn.get('transaction_type', 'Transaction'), 'financial')

    return {'nodes': list(nodes.values()), 'edges': edges}


def print_entity_graph(graph: dict):
    edge_sym = {'ownership': '──▶', 'directorship': '- -▶',
                'beneficial': '~~~▶', 'financial': '═══▶'}
    type_tag = {'company': '[CO]', 'person': '[PE]', 'bank': '[BK]'}

    print('\n' + '='*65)
    print('   ENTITY RELATIONSHIP GRAPH')
    print('='*65)
    print(f'  Nodes ({len(graph["nodes"])}):')
    for n in graph['nodes']:
        tag = type_tag.get(n['type'], '[??]')
        print(f'    {tag} {n["label"]:<40} risk={n["risk"]}')
        if n['data'].get('pan'): print(f'         PAN: {n["data"]["pan"]}')
        if n['data'].get('din'): print(f'         DIN: {n["data"]["din"]}')

    print(f'\n  Edges ({len(graph["edges"])}):')
    nodes_by_id = {n['id']: n for n in graph['nodes']}
    for e in graph['edges']:
        sym       = edge_sym.get(e['type'], '──▶')
        from_lbl  = nodes_by_id.get(e['from'], {}).get('label', e['from'])[:28]
        to_lbl    = nodes_by_id.get(e['to'],   {}).get('label', e['to'])[:28]
        print(f'    {from_lbl:<28} {sym} {to_lbl:<28}  [{e["label"]}]')
    print('='*65)


ENTITY_GRAPH = build_entity_graph(FRAUD_OCR_RESULTS)
print_entity_graph(ENTITY_GRAPH)

graph_file = OUTPUT_DIR / 'entity_graph.json'
with open(graph_file, 'w') as f:
    json.dump(ENTITY_GRAPH, f, indent=2)
print(f'\n  Saved: {graph_file}')


   ENTITY RELATIONSHIP GRAPH
  Nodes (7):
    [CO] MANIKANDAN CONSTRUCTIONS PVT LTD         risk=low
         PAN: AAKCM1234C
    [PE] D MANIKANDAN                             risk=low
         PAN: BNZPM2501F
         DIN: 106712345
    [CO] RAJESH INFRA PVT LTD                     risk=medium
    [PE] S RAJESHWARI                             risk=low
         PAN: ZZZRZ8765K
         DIN: 07823456
    [CO] DURAISAMY HOLDINGS                       risk=low
         PAN: DDDH7890C
    [CO] RAJESH INFRA PVT LTD                     risk=medium
    [CO] DURAISAMY HOLDINGS                       risk=medium

  Edges (6):
    D MANIKANDAN                 ──▶ MANIKANDAN CONSTRUCTIONS PVT  [Director (50%)]
    D MANIKANDAN                 - -▶ RAJESH INFRA PVT LTD          [Director]
    S RAJESHWARI                 ──▶ MANIKANDAN CONSTRUCTIONS PVT  [Director (50%)]
    DURAISAMY HOLDINGS           ──▶ MANIKANDAN CONSTRUCTIONS PVT  [Owns 20%]
    MANIKANDAN CONSTRUCTIONS PVT ═══▶ RAJESH INFRA

In [31]:
# ═══════════════════════════════════════════════════════════
# BENEFICIAL OWNERSHIP CHAIN TRAVERSAL
# Computes direct + indirect effective ownership % per person.
# This is the core of the beneficial ownership fraud pattern.
# ═══════════════════════════════════════════════════════════

def compute_ownership_chains(graph: dict) -> list:
    nodes_by_id = {n['id']: n for n in graph['nodes']}
    edges       = graph['edges']

    def parse_pct(label: str) -> float:
        m = _re.search(r'(\d+(?:\.\d+)?)\s*%', label)
        return float(m.group(1)) if m else 0.0

    # Direct person → MAIN_CO ownership
    direct_ownership = {}
    for e in edges:
        if e['to'] == 'MAIN_CO' and e['type'] == 'ownership':
            node = nodes_by_id.get(e['from'], {})
            if node.get('type') == 'person':
                direct_ownership[e['from']] = parse_pct(e['label'])

    # Company → MAIN_CO ownership, then person → that company via beneficial
    indirect_ownership = {}
    for e_co in edges:
        if e_co['to'] == 'MAIN_CO' and e_co['type'] == 'ownership':
            co_node = nodes_by_id.get(e_co['from'], {})
            if co_node.get('type') != 'company':
                continue
            co_pct = parse_pct(e_co['label'])
            for e_ben in edges:
                if e_ben['to'] == e_co['from'] and e_ben['type'] == 'beneficial':
                    pid = e_ben['from']
                    indirect_ownership.setdefault(pid, []).append({
                        'via_entity': co_node['label'],
                        'entity_pct': co_pct,
                        'effective_pct': co_pct,
                    })

    all_persons = set(direct_ownership.keys()) | set(indirect_ownership.keys())
    chains = []
    for pid in all_persons:
        node         = nodes_by_id.get(pid, {})
        direct_pct   = direct_ownership.get(pid, 0.0)
        indirect_lst = indirect_ownership.get(pid, [])
        total_indirect = sum(i['effective_pct'] for i in indirect_lst)
        chains.append({
            'person':           node.get('label', pid),
            'din':              node.get('data', {}).get('din'),
            'pan':              node.get('data', {}).get('pan'),
            'direct_pct':       direct_pct,
            'indirect_pct':     total_indirect,
            'total_pct':        direct_pct + total_indirect,
            'indirect_chains':  indirect_lst,
        })
    chains.sort(key=lambda x: x['total_pct'], reverse=True)
    return chains


OWNERSHIP_CHAINS = compute_ownership_chains(ENTITY_GRAPH)

print('\n' + '='*65)
print('   BENEFICIAL OWNERSHIP ANALYSIS')
print('='*65)
for chain in OWNERSHIP_CHAINS:
    print(f'\n  Person      : {chain["person"]}')
    print(f'  DIN         : {chain["din"]}  |  PAN: {chain["pan"]}')
    print(f'  Direct %    : {chain["direct_pct"]}%')
    print(f'  Indirect %  : {chain["indirect_pct"]}%')
    print(f'  Total %     : {chain["total_pct"]}%  <-- effective ownership')
    for ic in chain['indirect_chains']:
        print(f'    via {ic["via_entity"]} ({ic["entity_pct"]}% stake)')
print('='*65)


   BENEFICIAL OWNERSHIP ANALYSIS

  Person      : S RAJESHWARI
  DIN         : 07823456  |  PAN: ZZZRZ8765K
  Direct %    : 50.0%
  Indirect %  : 0%
  Total %     : 50.0%  <-- effective ownership

  Person      : D MANIKANDAN
  DIN         : 106712345  |  PAN: BNZPM2501F
  Direct %    : 50.0%
  Indirect %  : 0%
  Total %     : 50.0%  <-- effective ownership


In [32]:
# ═══════════════════════════════════════════════════════════
# PEP CHECK
# Reads from OCR-extracted PEP declaration document.
# Production: also call ComplyAdvantage / World-Check API.
# ═══════════════════════════════════════════════════════════

def check_pep(fraud_results: list) -> dict:
    pep_doc = next(
        (d for d in fraud_results
         if d['doc_type'] == 'PEP_DECLARATION' and d.get('status') == 'EXTRACTED'),
        None
    )
    if not pep_doc:
        return {'overall_flag': 'NOT_CHECKED', 'pep_found': 0,
                'reason': 'PEP declaration not submitted'}

    declarations = pep_doc['fields'].get('declarations', [])
    pep_persons  = [d for d in declarations
                   if 'NOT' not in d.get('pep_status', 'NOT A PEP').upper()]
    clean        = [d for d in declarations
                   if 'NOT' in d.get('pep_status', 'NOT A PEP').upper()]

    return {
        'total_checked': len(declarations),
        'pep_found':     len(pep_persons),
        'clean':         len(clean),
        'pep_persons':   pep_persons,
        'clean_persons': [d['name'] for d in clean],
        'overall_flag':  'FLAGGED' if pep_persons else 'CLEAR',
        'source':        'PEP_DECLARATION_DOCUMENT',
    }


PEP_RESULT = check_pep(FRAUD_OCR_RESULTS)

print('\n' + '='*65)
print('   PEP CHECK')
print('='*65)
print(f'  Overall Flag    : {PEP_RESULT["overall_flag"]}')
print(f'  Persons Checked : {PEP_RESULT["total_checked"]}')
print(f'  PEP Found       : {PEP_RESULT["pep_found"]}')
if PEP_RESULT.get('clean_persons'):
    print(f'  Clean           : {", ".join(PEP_RESULT["clean_persons"])}')
if PEP_RESULT.get('pep_persons'):
    print(f'  PEP FLAGGED     : {[p["name"] for p in PEP_RESULT["pep_persons"]]}')
print('='*65)


   PEP CHECK
  Overall Flag    : CLEAR
  Persons Checked : 2
  PEP Found       : 0
  Clean           : D MANIKANDAN, S RAJESHWARI


In [33]:
# ═══════════════════════════════════════════════════════════
# RPT RISK SCORING
# Rule-based scoring on related party transactions.
# ═══════════════════════════════════════════════════════════

RPT_RULES = {
    'COMMON_DIRECTOR':  {'weight': 30, 'desc': 'Common director between borrower and related party'},
    'BENEFICIAL_OWNER': {'weight': 25, 'desc': 'Beneficial owner with indirect stake'},
    'LARGE_AMOUNT':     {'weight': 20, 'desc': 'Transaction exceeds Rs. 25 Lakhs'},
    'SUB_CONTRACT':     {'weight': 15, 'desc': 'Sub-contracting to related party'},
    'INTER_CO_DEPOSIT': {'weight': 10, 'desc': 'Inter-company deposit or loan'},
    'NO_APPROVAL':      {'weight': 40, 'desc': 'Transaction without proper approval'},
}

def analyse_rpt(fraud_results: list) -> dict:
    rpt_doc = next(
        (d for d in fraud_results
         if d['doc_type'] == 'RPT_DOCUMENT' and d.get('status') == 'EXTRACTED'),
        None
    )
    if not rpt_doc:
        return {'overall_risk': 'NOT_CHECKED', 'total_transactions': 0}

    transactions = rpt_doc['fields'].get('related_party_transactions', [])
    flagged      = []
    max_score    = 0

    for txn in transactions:
        flags    = []
        score    = 0
        rel      = txn.get('relationship', '').lower()
        typ      = txn.get('transaction_type', '').lower()
        amt_str  = txn.get('amount', '')
        approval = txn.get('approval', '').lower()

        if 'director' in rel:
            flags.append('COMMON_DIRECTOR');    score += RPT_RULES['COMMON_DIRECTOR']['weight']
        if 'beneficial' in rel:
            flags.append('BENEFICIAL_OWNER');   score += RPT_RULES['BENEFICIAL_OWNER']['weight']
        if 'sub-contract' in typ or 'subcontract' in typ:
            flags.append('SUB_CONTRACT');        score += RPT_RULES['SUB_CONTRACT']['weight']
        if 'deposit' in typ or 'loan' in typ:
            flags.append('INTER_CO_DEPOSIT');    score += RPT_RULES['INTER_CO_DEPOSIT']['weight']

        amounts = _re.findall(r'[\d]+', amt_str.replace(',', ''))
        if amounts and int(amounts[0]) > 2500000:
            flags.append('LARGE_AMOUNT');        score += RPT_RULES['LARGE_AMOUNT']['weight']

        if approval and ('approval' not in approval or 'obtained' not in approval):
            flags.append('NO_APPROVAL');         score += RPT_RULES['NO_APPROVAL']['weight']

        risk_level = 'HIGH' if score >= 50 else ('MEDIUM' if score >= 25 else 'LOW')
        max_score  = max(max_score, score)

        flagged.append({
            'related_party': txn.get('related_party'),
            'risk_score':    score,
            'risk_level':    risk_level,
            'flags':         flags,
            'amount':        amt_str,
            'flag_reasons':  [RPT_RULES[f]['desc'] for f in flags if f in RPT_RULES],
        })

    overall = 'HIGH' if max_score >= 50 else ('MEDIUM' if max_score >= 25 else 'LOW')
    return {
        'total_transactions': len(transactions),
        'flagged':            flagged,
        'max_risk_score':     max_score,
        'overall_risk':       overall,
    }


RPT_RESULT = analyse_rpt(FRAUD_OCR_RESULTS)

print('\n' + '='*65)
print('   RPT RISK ANALYSIS')
print('='*65)
print(f'  Total Transactions : {RPT_RESULT["total_transactions"]}')
print(f'  Overall Risk       : {RPT_RESULT["overall_risk"]}')
print(f'  Max Score          : {RPT_RESULT["max_risk_score"]}/100')
for txn in RPT_RESULT.get('flagged', []):
    print(f'\n  Party      : {txn["related_party"]}')
    print(f'  Score      : {txn["risk_score"]}  |  Level: {txn["risk_level"]}')
    print(f'  Flags      : {", ".join(txn["flags"])}')
    for reason in txn.get('flag_reasons', []):
        print(f'    → {reason}')
print('='*65)


   RPT RISK ANALYSIS
  Total Transactions : 2
  Overall Risk       : HIGH
  Max Score          : 65/100

  Party      : RAJESH INFRA PVT LTD
  Score      : 65  |  Level: HIGH
  Flags      : COMMON_DIRECTOR, SUB_CONTRACT, LARGE_AMOUNT
    → Common director between borrower and related party
    → Sub-contracting to related party
    → Transaction exceeds Rs. 25 Lakhs

  Party      : DURAISAMY HOLDINGS
  Score      : 35  |  Level: MEDIUM
  Flags      : BENEFICIAL_OWNER, INTER_CO_DEPOSIT
    → Beneficial owner with indirect stake
    → Inter-company deposit or loan


In [34]:
# ═══════════════════════════════════════════════════════════
# PERSON PROFILES
# Combines director info, PEP status, ownership %
# ═══════════════════════════════════════════════════════════

def build_person_profiles(fraud_results, pep_result, ownership_chains) -> list:
    board_doc = next(
        (d for d in fraud_results
         if d['doc_type'] == 'BOARD_OF_DIRECTORS' and d.get('status') == 'EXTRACTED'),
        None
    )
    if not board_doc:
        return []

    pep_names       = {p['name'].upper() for p in pep_result.get('pep_persons', [])}
    ownership_by_name = {o['person'].upper(): o for o in ownership_chains}

    profiles = []
    for director in board_doc['fields'].get('directors', []):
        name     = director.get('name', '')
        o        = ownership_by_name.get(name.upper(), {})
        others_raw = director.get('other_directorships', '')
        other_cos  = []
        if others_raw and others_raw.upper() not in ('NIL', 'NONE', 'NA', ''):
            other_cos = [x.strip() for x in _re.split(r'[;|,]', others_raw) if x.strip()]

        is_pep     = name.upper() in pep_names
        risk_score = 50 if is_pep else (30 if other_cos else 10)

        profiles.append({
            'name':               name,
            'role':               director.get('designation', 'Director'),
            'din':                director.get('din'),
            'pan':                director.get('pan'),
            'dob':                director.get('dob'),
            'address':            director.get('address'),
            'nationality':        director.get('nationality'),
            'shareholding':       director.get('shareholding'),
            'pep':                is_pep,
            'sanctions':          False,
            'other_directorships':other_cos,
            'direct_pct':         o.get('direct_pct', 0),
            'indirect_pct':       o.get('indirect_pct', 0),
            'total_pct':          o.get('total_pct', 0),
            'risk_score':         risk_score,
        })
    return profiles


PERSON_PROFILES = build_person_profiles(FRAUD_OCR_RESULTS, PEP_RESULT, OWNERSHIP_CHAINS)

print('\n' + '='*65)
print('   PERSON PROFILES')
print('='*65)
for p in PERSON_PROFILES:
    print(f"\n  {p['name']}  ({p['role']})")
    print(f"  DIN: {p['din']}  |  PAN: {p['pan']}  |  DOB: {p['dob']}")
    print(f"  Ownership: {p['direct_pct']}% direct  +  {p['indirect_pct']}% indirect  =  {p['total_pct']}% total")
    print(f"  PEP: {'YES — FLAGGED' if p['pep'] else 'NO'}  |  Sanctions: {'YES' if p['sanctions'] else 'NO'}")
    print(f"  Risk Score: {p['risk_score']}/100")
    if p['other_directorships']:
        print(f"  Other Directorships: {', '.join(p['other_directorships'])}")
print('='*65)


   PERSON PROFILES

  D MANIKANDAN  (Managing Director)
  DIN: 106712345  |  PAN: BNZPM2501F  |  DOB: 16/07/1986
  Ownership: 50.0% direct  +  0% indirect  =  50.0% total
  PEP: NO  |  Sanctions: NO
  Risk Score: 30/100
  Other Directorships: RAJESH INFRA PVT LTD (Director since 2018)

  S RAJESHWARI  (Whole-Time Director)
  DIN: 07823456  |  PAN: ZZZRZ8765K  |  DOB: 22/03/1989
  Ownership: 50.0% direct  +  0% indirect  =  50.0% total
  PEP: NO  |  Sanctions: NO
  Risk Score: 10/100


In [35]:
# ═══════════════════════════════════════════════════════════
# NLP FRAUD RISK SUMMARY — qwen2.5:3b via Ollama
# Sends all extracted fraud signals to the model.
# Returns a list of 6-7 point-wise risk sentences.
# ═══════════════════════════════════════════════════════════

def generate_nlp_summary(entity_graph, pep_result, rpt_result,
                         ownership_chains, compliance_verdict) -> list:
    context = {
        'company':          'MANIKANDAN CONSTRUCTIONS PVT LTD',
        'graph_nodes':      len(entity_graph['nodes']),
        'graph_edges':      len(entity_graph['edges']),
        'nodes':            [{'label': n['label'], 'type': n['type']} for n in entity_graph['nodes']],
        'pep_flag':         pep_result.get('overall_flag'),
        'pep_count':        pep_result.get('pep_found', 0),
        'rpt_risk':         rpt_result.get('overall_risk'),
        'rpt_transactions': rpt_result.get('total_transactions', 0),
        'rpt_flags':        [t.get('flags') for t in rpt_result.get('flagged', [])],
        'ownership_chains': ownership_chains,
        'compliance':       compliance_verdict,
    }

    prompt = f"""You are a financial fraud analyst at an Indian NBFC reviewing a loan application.
Based on the data below, write a risk summary as a JSON array of 6-7 clear English sentences.
Each sentence is one item in the array. The last sentence must be the overall recommendation.
Return ONLY a valid JSON array of strings. No markdown, no explanation.

DATA:
{json.dumps(context, indent=2)}

Format: ["Point 1.", "Point 2.", "Point 3.", "Point 4.", "Point 5.", "Point 6.", "Recommendation."]"""

    try:
        resp = httpx.post(
            OLLAMA_URL,
            json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.2}},
            timeout=120.0
        )
        raw  = resp.json()['response'].strip()
        raw  = _re.sub(r'<think>.*?</think>', '', raw, flags=_re.DOTALL).strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            if raw.startswith('json'): raw = raw[4:]
        points = json.loads(raw.strip())
        return points if isinstance(points, list) else [raw]
    except Exception as e:
        return [f'NLP summary failed: {e}. Review data manually.']


# Load compliance verdict from Stage 2 if available
compliance_verdict = 'UNKNOWN'
cc_file = OUTPUT_DIR / 'cross_check_report.json'
if cc_file.exists():
    with open(cc_file) as f:
        compliance_verdict = json.load(f).get('verdict', 'UNKNOWN')

print('Generating NLP summary via qwen2.5:3b (may take ~30s)...')
NLP_SUMMARY = generate_nlp_summary(
    ENTITY_GRAPH, PEP_RESULT, RPT_RESULT, OWNERSHIP_CHAINS, compliance_verdict
)

print('\n' + '='*65)
print('   NLP FRAUD RISK SUMMARY (qwen2.5:3b)')
print('='*65)
for i, pt in enumerate(NLP_SUMMARY, 1):
    print(f'  {i}. {pt}')
print('='*65)

Generating NLP summary via qwen2.5:3b (may take ~30s)...

   NLP FRAUD RISK SUMMARY (qwen2.5:3b)
  1. NLP summary failed: Extra data: line 2 column 1 (char 790). Review data manually.


In [36]:
# ═══════════════════════════════════════════════════════════
# FINAL FRAUD RISK SCORE
# Combines all signals into a 0–100 composite score.
# ═══════════════════════════════════════════════════════════

def compute_risk_score(pep_result, rpt_result, entity_graph, compliance_verdict) -> dict:
    score = 0
    flags = []

    if pep_result.get('pep_found', 0) > 0:
        score += 40;  flags.append('PEP_DETECTED')

    rpt_score = min(rpt_result.get('max_risk_score', 0), 30)
    score += rpt_score
    if rpt_result.get('overall_risk') in ('HIGH', 'MEDIUM'):
        flags.append(f"RPT_RISK_{rpt_result['overall_risk']}")

    if len(entity_graph.get('nodes', [])) > 5:
        score += 10;  flags.append('COMPLEX_ENTITY_STRUCTURE')

    if len(entity_graph.get('edges', [])) > 7:
        score += 5;   flags.append('HIGH_RELATIONSHIP_COUNT')

    if compliance_verdict == 'FAIL':
        score += 15;  flags.append('COMPLIANCE_FAILED')

    score = min(score, 100)
    level = 'HIGH' if score >= 70 else ('MEDIUM' if score >= 40 else 'LOW')

    recs = {
        'HIGH':   'REJECT — High fraud risk. Do not proceed.',
        'MEDIUM': 'ENHANCED_DUE_DILIGENCE — Deeper investigation required before sanctioning.',
        'LOW':    'STANDARD_DUE_DILIGENCE — Proceed with normal periodic monitoring.',
    }
    return {
        'risk_score':     score,
        'risk_level':     level,
        'flags':          flags,
        'recommendation': recs[level],
    }


RISK_SCORE = compute_risk_score(PEP_RESULT, RPT_RESULT, ENTITY_GRAPH, compliance_verdict)

print('\n' + '='*65)
print('   FINAL FRAUD DETECTION REPORT')
print('='*65)
print(f'  Company        : MANIKANDAN CONSTRUCTIONS PVT LTD')
print(f'  PAN            : AAKCM1234C')
print(f'  Risk Score     : {RISK_SCORE["risk_score"]}/100')
print(f'  Risk Level     : {RISK_SCORE["risk_level"]}')
print(f'  Recommendation : {RISK_SCORE["recommendation"]}')
print(f'  Active Flags   : {", ".join(RISK_SCORE["flags"]) or "None"}')
print(f'  Compliance     : {compliance_verdict}')
print(f'  Graph          : {len(ENTITY_GRAPH["nodes"])} nodes, {len(ENTITY_GRAPH["edges"])} edges')
print(f'  Ownership Chains: {len(OWNERSHIP_CHAINS)} persons traced')
print(f'  PEP Status     : {PEP_RESULT["overall_flag"]}')
print(f'  RPT Risk       : {RPT_RESULT["overall_risk"]}')
print('='*65)

FRAUD_REPORT = {
    'timestamp':          datetime.now().isoformat(),
    'company':            'MANIKANDAN CONSTRUCTIONS PVT LTD',
    'pan':                'AAKCM1234C',
    'risk_score':         RISK_SCORE['risk_score'],
    'risk_level':         RISK_SCORE['risk_level'],
    'recommendation':     RISK_SCORE['recommendation'],
    'flags':              RISK_SCORE['flags'],
    'entity_graph':       {'nodes': len(ENTITY_GRAPH['nodes']), 'edges': len(ENTITY_GRAPH['edges'])},
    'ownership_chains':   OWNERSHIP_CHAINS,
    'pep_check':          PEP_RESULT,
    'rpt_analysis':       RPT_RESULT,
    'persons':            PERSON_PROFILES,
    'nlp_summary':        NLP_SUMMARY,
    'compliance_verdict': compliance_verdict,
}

report_file = OUTPUT_DIR / 'fraud_detection_report.json'
with open(report_file, 'w') as f:
    json.dump(FRAUD_REPORT, f, indent=2)
print(f'\n  Saved: {report_file}')


   FINAL FRAUD DETECTION REPORT
  Company        : MANIKANDAN CONSTRUCTIONS PVT LTD
  PAN            : AAKCM1234C
  Risk Score     : 40/100
  Risk Level     : MEDIUM
  Recommendation : ENHANCED_DUE_DILIGENCE — Deeper investigation required before sanctioning.
  Active Flags   : RPT_RISK_HIGH, COMPLEX_ENTITY_STRUCTURE
  Compliance     : UNKNOWN
  Graph          : 7 nodes, 6 edges
  Ownership Chains: 2 persons traced
  PEP Status     : CLEAR
  RPT Risk       : HIGH

  Saved: ocr_output\fraud_detection_report.json
